# Module 07: Quantization Basics
## ECBS5200 Pre-Work

**Goal:** Build intuition for what quantization is and why it matters.

We will NOT do actual model quantization here — that is Week 5. Instead, we will:
1. See how number formats affect storage size using raw numpy arrays
2. Measure the reconstruction error when we reduce precision
3. Connect this to a real model (ModernBERT-base) to see the practical implications

**Requirements:** CPU only. No GPU needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Part 1: Number Formats and Memory

Neural network weights are just arrays of numbers. Let's create a fake set of
"weights" and see how the storage format affects size.

In [ ]:
# Create 1 million random "weights" — typical values for a neural network
# Weights are usually small numbers centered around zero
np.random.seed(42)
weights_fp32 = np.random.randn(1_000_000).astype(np.float32)

print(f"Data type: {weights_fp32.dtype}")
print(f"Number of weights: {len(weights_fp32):,}")
print(f"Size in memory: {weights_fp32.nbytes:,} bytes ({weights_fp32.nbytes / 1e6:.1f} MB)")
print(f"Bytes per number: {weights_fp32.itemsize}")

Each FP32 number takes **4 bytes**. One million weights = 4 MB. Simple.

Now let's convert to FP16 (half precision):

In [ ]:
weights_fp16 = weights_fp32.astype(np.float16)

print(f"Data type: {weights_fp16.dtype}")
print(f"Size in memory: {weights_fp16.nbytes:,} bytes ({weights_fp16.nbytes / 1e6:.1f} MB)")
print(f"Bytes per number: {weights_fp16.itemsize}")
print(f"\nSize reduction: {weights_fp32.nbytes / weights_fp16.nbytes:.0f}x smaller")

Half the bits, half the memory. Now let's go further — INT8.

INT8 uses integers from -128 to 127, so we need to **scale** our floating-point
values into that range. This is the core of quantization.

In [ ]:
# Simple min-max quantization to INT8
w_min = weights_fp32.min()
w_max = weights_fp32.max()

# Scale to 0-255 range, then shift to -128..127
scale = (w_max - w_min) / 255.0
zero_point = np.round(-w_min / scale).astype(np.int8)

weights_int8 = np.clip(
    np.round((weights_fp32 - w_min) / scale) - 128, -128, 127
).astype(np.int8)

print(f"Data type: {weights_int8.dtype}")
print(f"Size in memory: {weights_int8.nbytes:,} bytes ({weights_int8.nbytes / 1e6:.1f} MB)")
print(f"Bytes per number: {weights_int8.itemsize}")
print(f"\nSize reduction vs FP32: {weights_fp32.nbytes / weights_int8.nbytes:.0f}x smaller")

## Summary So Far

| Format | Bytes per Number | Total Size (1M weights) | Reduction |
|--------|:---:|:---:|:---:|
| FP32 | 4 | 4.0 MB | 1x |
| FP16 | 2 | 2.0 MB | 2x |
| INT8 | 1 | 1.0 MB | 4x |

In [ ]:
# Let's show this visually
formats = ["FP32\n(32-bit)", "FP16\n(16-bit)", "INT8\n(8-bit)"]
sizes_mb = [
    weights_fp32.nbytes / 1e6,
    weights_fp16.nbytes / 1e6,
    weights_int8.nbytes / 1e6,
]
colors = ["#2196F3", "#FF9800", "#4CAF50"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(formats, sizes_mb, color=colors, edgecolor="black", linewidth=0.5)
for bar, size in zip(bars, sizes_mb):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        f"{size:.1f} MB",
        ha="center",
        va="bottom",
        fontweight="bold",
        fontsize=12,
    )
ax.set_ylabel("Memory (MB)")
ax.set_title("Memory Usage by Number Format (1M weights)")
ax.set_ylim(0, 5)
plt.tight_layout()
plt.show()

## Part 2: Reconstruction Error

When we quantize to INT8, we lose information. How much? Let's convert
the INT8 values **back** to floating point and compare to the originals.

In [ ]:
# Dequantize: convert INT8 back to float
weights_reconstructed = (weights_int8.astype(np.float32) + 128) * scale + w_min

# Measure the error
error = weights_fp32 - weights_reconstructed
mean_abs_error = np.mean(np.abs(error))
max_abs_error = np.max(np.abs(error))
rmse = np.sqrt(np.mean(error**2))

print("Reconstruction Error (INT8 -> FP32):")
print(f"  Mean absolute error:  {mean_abs_error:.6f}")
print(f"  Max absolute error:   {max_abs_error:.6f}")
print(f"  RMSE:                 {rmse:.6f}")
print(f"\nFor context, the original weights range from {w_min:.4f} to {w_max:.4f}")
print(f"The mean absolute error is {mean_abs_error / (w_max - w_min) * 100:.3f}% of the full range")

The error is small — fractions of a percent of the value range. This is why
quantization works in practice: the rounding errors are tiny relative to the
scale of the values.

In [ ]:
# Visualize: original vs reconstructed distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: overlay histograms
axes[0].hist(weights_fp32, bins=100, alpha=0.6, label="Original (FP32)", color="#2196F3")
axes[0].hist(
    weights_reconstructed, bins=100, alpha=0.6, label="Reconstructed (INT8)", color="#4CAF50"
)
axes[0].set_xlabel("Weight Value")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution: Original vs Reconstructed")
axes[0].legend()

# Right: error distribution
axes[1].hist(error, bins=100, color="#FF5722", alpha=0.7)
axes[1].set_xlabel("Error (Original - Reconstructed)")
axes[1].set_ylabel("Count")
axes[1].set_title("Reconstruction Error Distribution")
axes[1].axvline(x=0, color="black", linestyle="--", linewidth=0.8)

plt.tight_layout()
plt.show()

**Left plot:** The distributions look nearly identical. This is the JPEG analogy —
you can barely tell the difference.

**Right plot:** The errors are small, centered around zero, and uniformly distributed.
This is the quantization noise that we accept in exchange for a 4x size reduction.

## Part 3: Real Model — ModernBERT-base

Let's connect this to a model we actually use in this course.

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained("answerdotai/ModernBERT-base")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model: ModernBERT-base")
print(f"Total parameters: {total_params:,}")

Now let's compute what this model would cost in each format:

In [ ]:
# Compute sizes for each format
size_fp32 = total_params * 4  # 4 bytes per param
size_fp16 = total_params * 2  # 2 bytes per param
size_int8 = total_params * 1  # 1 byte per param

print(f"{'Format':<10} {'Bytes/Param':<15} {'Model Size':<15} {'Reduction':<10}")
print("-" * 50)
print(f"{'FP32':<10} {'4':<15} {size_fp32 / 1e6:>8.1f} MB     {'1x':<10}")
print(f"{'FP16':<10} {'2':<15} {size_fp16 / 1e6:>8.1f} MB     {'2x':<10}")
print(f"{'INT8':<10} {'1':<15} {size_int8 / 1e6:>8.1f} MB     {'4x':<10}")

In [ ]:
# Visualize for the real model
fig, ax = plt.subplots(figsize=(8, 4))
formats = ["FP32\n(Full Precision)", "FP16\n(Half Precision)", "INT8\n(Quarter Precision)"]
sizes = [size_fp32 / 1e6, size_fp16 / 1e6, size_int8 / 1e6]
colors = ["#2196F3", "#FF9800", "#4CAF50"]

bars = ax.bar(formats, sizes, color=colors, edgecolor="black", linewidth=0.5)
for bar, size in zip(bars, sizes):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{size:.0f} MB",
        ha="center",
        va="bottom",
        fontweight="bold",
        fontsize=12,
    )
ax.set_ylabel("Model Size (MB)")
ax.set_title(f"ModernBERT-base ({total_params/1e6:.0f}M params) — Size by Precision")
ax.set_ylim(0, max(sizes) * 1.15)
plt.tight_layout()
plt.show()

## The Bottom Line

Quantization is one of the highest-leverage techniques in production ML engineering.
A 4x reduction in model size means:

- **4x less GPU memory** needed to serve the model
- **Cheaper hardware** — fit on a T4 instead of an A100
- **Lower latency** — less data moving through memory
- **Edge deployment** — run on phones and embedded devices

The cost? A tiny amount of numerical precision. For most tasks, this trade-off is
overwhelmingly worth it.

**You will implement real quantization in Week 5.** For now, the key intuition is:
same model, fewer bits, dramatic savings, small (usually acceptable) quality cost.

## Check Yourself

Before moving on, make sure you can answer these questions:

1. **What is a model weight?** What role do weights play in a neural network?
2. **How many bytes does a single FP32 number use?** What about FP16? INT8?
3. **If a model has 149M parameters in FP32, how large is it in megabytes?**
   What about in INT8?
4. **Why is quantization like JPEG compression?** What is the analogy?
5. **Name two scenarios where quantization is particularly valuable.**
6. **What is the main risk of quantization?** How would you check for it?